# Planned vs. real bus arrivals — interactive exploration

One question: **how long does a bus really take to get from each stop to the next, and how does
that compare to the published timetable?**

All the work happens in the `bus_times` package; this notebook only calls it. If you want to know
*how* the numbers are produced — in particular why actual arrival times have to be derived from
raw GPS rather than read from a field — see
`docs/superpowers/specs/2026-07-30-bus-arrival-analysis-design.md`.

**The data is patchy, and the charts say so.** Every chart marks where its own numbers are weak:
hatched bars and cells, dimmed axis labels, ride counts on every mark, and a caveat line along the
bottom. Read those before reading the shapes.

In [ ]:
import datetime

import pandas as pd

from bus_times import (
    aggregate_segments,
    elapsed_profiles,
    find_lines,
    load_line_data,
    plot_marey,
    plot_segment_hour_heatmap,
    plot_segment_times,
    quality_summary,
    resolve_line,
    segment_hour_matrix,
    stop_coverage,
)

pd.options.display.width = 200
pd.options.display.max_colwidth = 80

# SIRI history is short and the newest days are still being ingested, so work a few days back.
DATE_TO = datetime.date.today() - datetime.timedelta(days=3)
DATE_FROM = DATE_TO - datetime.timedelta(days=6)
MIN_SAMPLES = 3
DATE_FROM, DATE_TO

## Step 1 — find the line you want

A line number is not an identifier. Egged runs a line "15" in Jerusalem, Haifa, Eilat, Rehovot and
several other cities, and each direction of each is a separate route. `find_lines` lists all of
them so you can pick; `route_long_name` names the terminal stops, which is how you tell the cities
apart.

In [ ]:
find_lines(15, DATE_FROM, DATE_TO, agency_name='אגד')

`resolve_line` narrows that list to exactly one route and fails loudly if the description is still
ambiguous — picking the wrong direction by accident produces charts that look perfectly plausible
and are entirely wrong.

In [ ]:
line = resolve_line(15, DATE_FROM, DATE_TO,
                    agency_name='אגד', name_contains='ירושלים', direction=1)
line

## Step 2 — fetch

`load_line_data` pulls the planned timetable, samples rides evenly across days and departure hours,
fetches their GPS trails, and derives an arrival time per stop. It prints its coverage as it goes.

**This takes a few minutes.** GPS costs about 0.7 s per ride against the public API. Narrow
`hour_range` or lower `max_rides_per_hour` to go faster.

In [ ]:
stop_events, ride_segments = load_line_data(
    line, DATE_FROM, DATE_TO,
    hour_range=(7, 20),
    max_rides_per_hour=2,
)
stop_events.head()

## Step 3 — check the data before trusting it

Two columns on `stop_events` say how believable each derived arrival is:

- **`match_distance_m`** — how close the bus actually got to the stop. A few metres is a real match;
  200 m means we are guessing which stop it was at.
- **`resolution_s`** — the gap between the two GPS pings the arrival was interpolated from, i.e. the
  precision of that timestamp. A 20-second gap pins the arrival down; a five-minute gap does not.

`stop_coverage` rolls these up per stop, along with how often an arrival could be derived at all.

In [ ]:
coverage = stop_coverage(stop_events)
# The stops most likely to mislead you: rarely matched, or matched loosely.
coverage.nsmallest(8, 'coverage')[
    ['stop_sequence', 'stop_name', 'matched', 'rides', 'coverage',
     'match_distance_m', 'resolution_s']
]

`aggregate_segments` attaches a `confidence` verdict to every segment. Under-sampled segments are
**kept and flagged rather than dropped**: a segment missing from a chart is indistinguishable from a
segment that does not exist, which is the most misleading failure available here.

The verdicts, worst first: `implausible value`, `few samples`, `patchy coverage`,
`coarse GPS timing`, `loose stop match`, `ok`.

In [ ]:
aggregated = aggregate_segments(ride_segments, min_samples=MIN_SAMPLES)
print(quality_summary(aggregated))
aggregated['confidence'].value_counts()

In [ ]:
# Exactly which segments to distrust, and why.
aggregated.loc[~aggregated['is_reliable'],
               ['from_name', 'to_name', 'sample_count', 'coverage',
                'resolution_s', 'match_distance_m', 'confidence']]

## Chart 1 — where is the timetable optimistic?

Bars are the median measured duration per segment, whiskers the interquartile range, diamonds the
planned duration. Where the bar overshoots the diamond, that stretch of road takes longer than the
schedule allows.

**Reading the caveats:** a solid bar is trustworthy; a pale hatched bar is not, and the note beside
it gives the ride count and the reason. Every bar carries its `n=`, so no mark's weight of evidence
is hidden. The bottom line summarises how much of the route is well measured.

The median rather than the mean is deliberate: arrival times come from GPS, so an occasional ride is
minutes out, and one such ride moves a mean enough to flatten every other segment on the axis.

In [ ]:
plot_segment_times(aggregated, line.label, f'{DATE_FROM}..{DATE_TO}')

In [ ]:
# The worst offenders — restricted to segments whose numbers hold up.
reliable = aggregated[aggregated['is_reliable']]
reliable.assign(
    over_min=(reliable['actual_median_s'] - reliable['planned_duration_s']) / 60,
    ratio=reliable['actual_median_s'] / reliable['planned_duration_s'],
).nlargest(8, 'over_min')[['from_name', 'to_name', 'over_min', 'ratio', 'sample_count']]

## Chart 2 — the Marey time-space diagram

The most information-dense of the three. Each thin line is one ride's progress down the route; the
dashed line is the schedule. Read it like this:

- **steep** = the bus is moving well; **flat** = stuck
- lines to the *right* of the dashed reference are running behind schedule
- the **width of the fan** is the line's unreliability — a narrow fan is a predictable route

The x axis is minutes *elapsed since departure*, not clock time, which is what lets rides from
different hours and days sit on one axis.

**Reading the caveats:** stops the GPS rarely resolved are dimmed and italicised on the axis, with
their match rate shown. A trajectory still gets drawn through those stops, so without the marking
there would be nothing to distinguish measurement from interpolation.

In [ ]:
actual, planned = elapsed_profiles(stop_events)
plot_marey(actual, planned, line.label, f'{DATE_FROM}..{DATE_TO}', coverage=coverage)

## Chart 3 — which segments break down at rush hour?

Each cell is the median actual/planned duration ratio for one segment at one departure hour. Red is
slower than scheduled, blue quicker, and the neutral centre is exactly on schedule. A ratio rather
than a duration so a 30-second hop and a five-minute run share one colour scale.

**Reading the caveats:** three appearances, three states — solid means at least `MIN_SAMPLES` rides,
hatched means fewer, and blank means no ride produced a usable value at all. The number in each cell
is its ride count. Collapsing "one ride" and "no data" into the same blank cell would hide exactly
what you need to know.

Widen `hour_range` above to get more columns.

In [ ]:
matrix = segment_hour_matrix(ride_segments)
plot_segment_hour_heatmap(matrix, line.label, f'{DATE_FROM}..{DATE_TO}',
                          min_samples=MIN_SAMPLES)

In [ ]:
# How thin is the grid overall? Cells at 0 have no data; low counts are the hatched ones.
matrix.count.stack().value_counts().sort_index().head(10)

## What these numbers are not

Worth keeping in mind before quoting any of the above:

- **Arrival times are derived, not reported.** They are the moment of closest approach to a stop's
  coordinates, interpolated between GPS pings that arrive roughly once a minute — so about ±30 s per
  stop at best, and worse wherever `resolution_s` is large. Consecutive city stops are often less
  than a minute apart, so a *single* ride's short-segment duration is mostly noise. The aggregate
  views are the point.
- **The first segment is the least trustworthy.** Buses idle at the terminal, and although the origin
  stop resolves to the moment the bus left rather than its closest approach, that boundary is still
  the fuzziest one on the route.
- **Rides are sampled, not exhaustive.** Check the printed coverage line and the `sample_count`
  column before reading much into any single segment.
- **Stops the bus never came within 300 m of are dropped**, which costs the two segments either side.
  Those show up as reduced `coverage`, not as missing rows.

To save any figure without clipping its Hebrew labels, use `bus_times.save_figure(fig, path)` —
plain `fig.savefig` cuts off labels that sit outside the axes.